In [ ]:
import matplotlib as mpl
%matplotlib inline
# Set the font to Times New Roman or a similar serif font
mpl.rcParams['font.family'] = 'serif'
mpl.rcParams['font.serif'] = 'Times New Roman'
# Adjust the text sizes
mpl.rcParams['axes.labelsize'] = 10  # For X and Y axis labels
mpl.rcParams['axes.titlesize'] = 10  # For the plot title
mpl.rcParams['legend.fontsize'] = 10 # For the legend


from utils import *

In [ ]:
EJ_values = np.linspace(2, 4, 150)
Er_values = np.linspace(6.2,7.4, 150)
EC = 0.6
EL = 0.13
returned_tuple = get_EJ_Er_sweep_data(EJ_values, 
                                    Er_values,
                                    EC,
                                    EL,
                                    computational_state = [0,1],
                                    leakage_state = 2,
                                    )
with open('pickles/EJEr_sweep_01.pkl', 'wb') as file:
    pickle.dump(returned_tuple, file)

In [ ]:
EJ_values = np.linspace(2, 3.5, 150)
Er_values = np.linspace(6.75, 7.6, 150)
EC = 0.6
EL = 0.13

returned_tuple = get_EJ_Er_sweep_data(EJ_values, 
                                    Er_values,
                                    EC,
                                    EL,
                                    computational_state = [1,2],
                                    leakage_state = 0,
                                    )

with open('pickles/EJEr_sweep_12.pkl', 'wb') as file:
    pickle.dump(returned_tuple, file)


In [ ]:
# fig = plt.figure(figsize=(2*(3+3/8), 
#                             (3+3/8)/1.8*2))
fig = plt.figure(figsize=(2*(3+3/8)*1.3, 
                            (3+3/8)/1.8*2*1.5))
gs = gridspec.GridSpec(2, 3, width_ratios=[1.2, 1.2, 1], height_ratios=[1,1], wspace=0.2)
legend=True
line_styles = [
    (0,(1, 1)),
    (0,(3, 3)),
    (0,(3,2,1,2)),
    (0, (1,1,1,1,3,3)),
    (0, (1,1,1,1,1,1,3,3)),
    (0, (1,1,2,2,1,1,4,5)),
    (0, (5,5)),
    (0, (5,1,5,1))
]

#########################################################################################################
#
#
# Plotting the firt three subplots
#
#
#########################################################################################################
with open('pickles/EJEr_sweep_01.pkl', 'rb') as file:
    loaded_tuple = pickle.load(file)
transitions_to_0, transitions_to_1, transitions_to_2, Z1, Z2 = loaded_tuple
transitions = [transitions_to_0, transitions_to_1, transitions_to_2]
EJ_values = np.linspace(2, 4, 150)
Er_values = np.linspace(6.2,7.4, 150)
norm1 = LogNorm(vmin=1e-5,vmax=1e-2)
norm2 = LogNorm(vmin=1e-3,vmax=1)
computational_state=[0,1]
leakage_state=2
X, Y = np.meshgrid(EJ_values, Er_values)


line_style_counter = [[],[],[]]
next_line_style_idx = 0
def plot_transition_curves(initial_level):
    global next_line_style_idx
    if line_style_counter[initial_level] == []:
        for level, list_of_transitions in enumerate(transitions[initial_level]):
            if level % 2 == 1 - (initial_level%2) and np.max(list_of_transitions) > Er_values[0] and np.min(list_of_transitions) < Er_values[-1]:
                plt.plot(EJ_values, list_of_transitions, linestyle = line_styles[next_line_style_idx], linewidth=2,color = 'black',label = rf'$\omega_{{{initial_level},{level}}}$')
                line_style_counter[initial_level].append(next_line_style_idx)
                next_line_style_idx += 1
                next_line_style_idx = next_line_style_idx % len(line_styles)
    else:
        local_counter = 0
        for level, list_of_transitions in enumerate(transitions[initial_level]):
            if level % 2 == 1 - (initial_level%2) and np.max(list_of_transitions) > Er_values[0] and np.min(list_of_transitions) < Er_values[-1]:
                try:
                    plt.plot(EJ_values, list_of_transitions, linestyle = line_styles[line_style_counter[initial_level][local_counter]], linewidth=2,color = 'black',label = rf'$\omega_{{{initial_level},{level}}}$')
                except:
                    print(line_styles[line_style_counter[initial_level][local_counter]])
                local_counter += 1
                

################################################################################
# Heatmap about the diff of shift from the two computational states
################################################################################
ax0 = plt.subplot(gs[0,0])
plt.text(0.05, 1, '(a)', transform=plt.gca().transAxes, fontsize=12, fontweight='bold', va='top', color='black')
z1_plot = plt.pcolormesh(X, Y, Z1, shading='auto', cmap='inferno', norm=norm1)
plt.colorbar(z1_plot)

plot_transition_curves(computational_state[0])
plot_transition_curves(computational_state[1])

if legend:
    plt.legend(loc='lower left')
# plt.xlabel('EJ')
plt.ylabel('Er')
ax0.set_xlim([EJ_values[0], EJ_values[-1]])
ax0.set_ylim([Er_values[0], Er_values[-1]])
ax0.grid(which='major', color='grey', linestyle='-', linewidth=0.5)
ax0.minorticks_on()
ax0.grid(which='minor', color='grey', linestyle=':', linewidth=0.5)



################################################################################
# Heatmap about the diff of shift from the first computational state and the leakage state
################################################################################
ax1 = plt.subplot(gs[0,1])
plt.text(0.05, 1, '(b)', transform=plt.gca().transAxes, fontsize=12, fontweight='bold', va='top', color='black')
z2_plot = plt.pcolormesh(X, Y, Z2, shading='auto', cmap='inferno', norm=norm2)
plt.colorbar(z2_plot)

plot_transition_curves(computational_state[0])
plot_transition_curves(leakage_state)

if legend:
    plt.legend(loc='lower left')
# plt.xlabel('EJ')
# plt.ylabel('Er')
ax1.set_xlim([EJ_values[0], EJ_values[-1]])
ax1.set_ylim([Er_values[0], Er_values[-1]])
ax1.grid(which='major', color='grey', linestyle='-', linewidth=0.5)
ax1.minorticks_on()
ax1.grid(which='minor', color='grey', linestyle=':', linewidth=0.5)
# ax3.set_xticks([]) 
# ax3.set_yticks([])


################################################################################
# Additional subplot for overlay
################################################################################
ax2 = plt.subplot(gs[0,2])
plt.text(0.05, 1, '(c)', transform=plt.gca().transAxes, fontsize=12, fontweight='bold', va='top', color='black')
# Overlay of Z1 and Z2
plt.pcolormesh(X, Y, Z1, shading='auto', cmap='inferno', alpha=0.5, norm=norm1)
plt.pcolormesh(X, Y, Z2, shading='auto', cmap='inferno', alpha=0.5, norm=norm2)

plot_transition_curves(computational_state[0])
plot_transition_curves(computational_state[1])
plot_transition_curves(leakage_state)

# plt.xlabel('EJ')
# plt.ylabel('Er')
ax2.set_xlim([EJ_values[0], EJ_values[-1]])
ax2.set_ylim([Er_values[0], Er_values[-1]])
ax2.grid(which='major', color='grey', linestyle='-', linewidth=0.5)
ax2.minorticks_on()
ax2.grid(which='minor', color='grey', linestyle=':', linewidth=0.5)





#########################################################################################################
#
#
# Plotting the second three subplots
#
#
#########################################################################################################
with open('pickles/EJEr_sweep_12.pkl', 'rb') as file:
    loaded_tuple = pickle.load(file)
transitions_to_0, transitions_to_1, transitions_to_2, Z1, Z2 = loaded_tuple
transitions = [transitions_to_0, transitions_to_1, transitions_to_2]
EJ_values = np.linspace(2, 3.5, 150)
Er_values = np.linspace(6.75, 7.6, 150)
norm1 = LogNorm(vmin=1e-5,vmax=1e-3)
norm2 = LogNorm(vmin=5e-3,vmax=1)
computational_state=[1,2]
leakage_state=0
X, Y = np.meshgrid(EJ_values, Er_values)

line_style_counter = [[],[],[]]
next_line_style_idx = 0
def plot_transition_curves(initial_level):
    global next_line_style_idx
    if line_style_counter[initial_level] == []:
        for level, list_of_transitions in enumerate(transitions[initial_level]):
            if level % 2 == 1 - (initial_level%2) and np.max(list_of_transitions) > Er_values[0] and np.min(list_of_transitions) < Er_values[-1]:
                plt.plot(EJ_values, list_of_transitions, linestyle = line_styles[next_line_style_idx], linewidth=2,color = 'black',label = rf'$\omega_{{{initial_level},{level}}}$')
                line_style_counter[initial_level].append(next_line_style_idx)
                next_line_style_idx += 1
                next_line_style_idx = next_line_style_idx % len(line_styles)
    else:
        local_counter = 0
        for level, list_of_transitions in enumerate(transitions[initial_level]):
            if level % 2 == 1 - (initial_level%2) and np.max(list_of_transitions) > Er_values[0] and np.min(list_of_transitions) < Er_values[-1]:
                try:
                    plt.plot(EJ_values, list_of_transitions, linestyle = line_styles[line_style_counter[initial_level][local_counter]], linewidth=2,color = 'black',label = rf'$\omega_{{{initial_level},{level}}}$')
                except:
                    print(line_styles[line_style_counter[initial_level][local_counter]])
                local_counter += 1
                

################################################################################
# Heatmap about the diff of shift from the two computational states
################################################################################
ax0 = plt.subplot(gs[1,0])
plt.text(0.05, 1, '(d)', transform=plt.gca().transAxes, fontsize=12, fontweight='bold', va='top', color='black')
z1_plot = plt.pcolormesh(X, Y, Z1, shading='auto', cmap='inferno', norm=norm1)
plt.colorbar(z1_plot)

plot_transition_curves(computational_state[0])
plot_transition_curves(computational_state[1])

if legend:
    plt.legend(loc='lower left')
plt.xlabel('EJ')
plt.ylabel('Er')
ax0.set_xlim([EJ_values[0], EJ_values[-1]])
ax0.set_ylim([Er_values[0], Er_values[-1]])
ax0.grid(which='major', color='grey', linestyle='-', linewidth=0.5)
ax0.minorticks_on()
ax0.grid(which='minor', color='grey', linestyle=':', linewidth=0.5)



################################################################################
# Heatmap about the diff of shift from the first computational state and the leakage state
################################################################################
ax1 = plt.subplot(gs[1,1])
plt.text(0.05, 1, '(e)', transform=plt.gca().transAxes, fontsize=12, fontweight='bold', va='top', color='black')
z2_plot = plt.pcolormesh(X, Y, Z2, shading='auto', cmap='inferno', norm=norm2)
plt.colorbar(z2_plot)

plot_transition_curves(computational_state[0])
plot_transition_curves(leakage_state)

if legend:
    plt.legend(loc='lower left')
plt.xlabel('EJ')
# plt.ylabel('Er')
ax1.set_xlim([EJ_values[0], EJ_values[-1]])
ax1.set_ylim([Er_values[0], Er_values[-1]])
ax1.grid(which='major', color='grey', linestyle='-', linewidth=0.5)
ax1.minorticks_on()
ax1.grid(which='minor', color='grey', linestyle=':', linewidth=0.5)
# ax3.set_xticks([]) 
# ax3.set_yticks([])


################################################################################
# Additional subplot for overlay
################################################################################
ax2 = plt.subplot(gs[1,2])
plt.text(0.05, 1, '(f)', transform=plt.gca().transAxes, fontsize=12, fontweight='bold', va='top', color='black')
# Overlay of Z1 and Z2
plt.pcolormesh(X, Y, Z1, shading='auto', cmap='inferno', alpha=0.5, norm=norm1)
plt.pcolormesh(X, Y, Z2, shading='auto', cmap='inferno', alpha=0.5, norm=norm2)

plot_transition_curves(computational_state[0])
plot_transition_curves(computational_state[1])
plot_transition_curves(leakage_state)

plt.xlabel('EJ')
# plt.ylabel('Er')
ax2.set_xlim([EJ_values[0], EJ_values[-1]])
ax2.set_ylim([Er_values[0], Er_values[-1]])
ax2.grid(which='major', color='grey', linestyle='-', linewidth=0.5)
ax2.minorticks_on()
ax2.grid(which='minor', color='grey', linestyle=':', linewidth=0.5)

plt.savefig('fig03_sweep_EJ_Er.pdf', format='pdf', bbox_inches='tight')
plt.show()